**[🏠 Course Home](00_START_HERE.ipynb)** | **🤖 [AI-Native](../ai/20_test_promotion_heuristics_vs_bayes.ipynb)** | **🐍 [Python](../python/20_test_promotion_heuristics_vs_bayes.ipynb)** | **🔵 [R](../r/20_test_promotion_heuristics_vs_bayes.ipynb)** | **⬅️ [Prev: Ch 19: Verifying Flaky Fixes (SPRT & BF)](19_verifying_flaky_test_fixes.ipynb)** | **➡️ [Next: Ch 21: Framework Reliability Contracts](21_framework_reliability_and_tolerance_contracts.ipynb)**

*🧭 Part V: Production Reliability, CI/CD & Active Learning — Chapter 20 of 23*

---

# Chapter 20: CI/CD Test Promotion — Streak Heuristics ($N, M$) vs. Bayesian Filtering
### *The Gambler's Fallacy in CI, Survivorship Bias, and Dynamic Memory Promotion Gates*

---

## 1. What Are We Trying to Do?

In modern Continuous Integration (CI/CD) pipelines, automated tests are divided into two distinct tiers:
1. **Staging / Quarantine (Non-blocking)**: Runs on PRs but its failure does not prevent merging. Used to monitor new or newly-patched tests.
2. **Blocking Quality Gate**: Must pass 100% for code to be deployed. A failure halts the entire engineering pipeline.

The multi-million-dollar question for DevOps leaders is:
> **When is an unquarantined test reliable enough to be promoted from Staging to Blocking?**

---

## 2. The Standard Industry Heuristic: The $(N, M)$ Streak Rule

Many engineering organizations adopt a rule that looks like this:
> *"A test is promoted to blocking if it has achieved a historical streak of at least $N = 50$ consecutive passes at some point, and its current streak is at least $M = 10$ consecutive passes."*

At first glance, this rule feels rigorous: 50 passes in a row sounds like an impressive achievement!
However, mathematical and probabilistic analysis reveals **four fatal vulnerabilities** in this heuristic.

---

## 3. The 4 Hidden Traps of the Streak Heuristic

```
                      THE FOUR VULNERABILITIES OF STREAKS
                      
   1. The Geometric Trap               2. Total Amnesia
      A 5% flaky test has a 60% chance    Completely ignores failures that
      of achieving a 50-streak over time. happened 1 run before the streak.
      
   3. The Cold Streak Illusion         4. No Economic Calibration
      In binomial processes, streaks      Treats the cost of a false promotion
      of passes happen naturally by luck. the same as the cost of waiting.
```

### Trap 1: The Geometric Trap & Survivorship Bias
* Suppose a test has a true, permanent flakiness rate of **$5\%$** ($p = 0.05$). It is a broken test.
* What is the probability that it passes 50 times in a row on any given attempt?
  $$P(\text{Streak of 50}) = (1 - 0.05)^{50} = (0.95)^{50} \approx \mathbf{7.69\%}$$
* If this test runs 500 times in staging over a couple of months:
  $$P(\text{Achieves at least one 50-streak in 500 runs}) \approx \mathbf{58.4\%}!$$
* **The Takeaway**: A test that is permanently broken has an almost **60% chance** of meeting the promotion criteria purely through lucky randomness!

### Trap 2: Structural Amnesia
* Imagine Test A: Fails 10 times in a row, then gets lucky and passes 10 times in a row.
* Imagine Test B: Passes 1,000 times in a row with zero failures.
* Under the current streak requirement ($M = 10$), **both tests look identical to the heuristic!** The rule has complete amnesia regarding what happened 11 runs ago.

### Trap 3: The Cold Streak Illusion
* In baseball, a $.200$ hitter routinely experiences a slump where they go 0-for-10 at the plate.
* In probability, random events cluster. Streaks of passes and streaks of failures naturally appear in any Bernoulli process. Counting streaks measures **luck**, not underlying system stability!

---

## 4. The Bayesian Solution: Dynamic Discount State Machine

Instead of counting streaks, a rigorous Bayesian CI/CD governance engine maintains a **continuous health meter** using the discounted Beta-Binomial filter (as introduced in Chapter 15).

```
                        THE BAYESIAN PROMOTION GATE
                        
     Incoming CI Telemetry:  [Pass, Pass, Pass, Fail, Pass, ...]
                                          |
                                          v
     Exponential Memory Filter:  Multiply past by γ=0.98, add new data
                                          |
                                          v
     Posterior Belief:           P(Flake Rate ≤ 1.0% | Telemetry)
                                          |
                   +----------------------+----------------------+
                   |                                             |
           If Confidence ≥ 95%                          If Confidence < 95%
                   |                                             |
                   v                                             v
        [PROMOTE TO BLOCKING]                         [RETAIN IN STAGING]
```

---

### 4.1 Mathematical Formulation of the Discounted Beta-Binomial Filter

Let the true latent failure probability of test $i$ at batch/run $t$ be $\theta_t \in [0, 1]$. We model test executions under a **Beta-Binomial state-space filter** with exponential memory discount factor $\gamma \in (0, 1]$ (typically $\gamma = 0.98$ for CI pipelines, providing an effective memory window of $N_{\text{eff}} = \frac{1}{1-\gamma} \approx 50$ runs, or $\gamma = 0.995$ for long-horizon stability):

$$
\alpha_t = \gamma \alpha_{t-1} + k_t, \qquad \beta_t = \gamma \beta_{t-1} + (m_t - k_t)
$$

where:
* $m_t$ is the number of test invocations in batch $t$ (or $m_t = 1$ for per-run streaming telemetry).
* $k_t \in \{0, \dots, m_t\}$ is the number of observed failures in batch $t$.
* $(m_t - k_t)$ is the number of observed successful passes.
* $\gamma \in (0, 1]$ discounts past evidence, ensuring recent regressions register rapidly while ancient history gracefully fades.

#### The Skeptical Quarantine Prior
When a new test is introduced or a broken test enters quarantine following a patch, we do not assume it is innocent. We initialize the filter with a **skeptical quarantine prior**:

$$
\theta_0 \sim \text{Beta}(\alpha_0 = 1.0, \; \beta_0 = 19.0) \implies \mathbb{E}[\theta_0] = \frac{1.0}{1.0 + 19.0} = 0.050 \; (5.0\%)
$$

This prior asserts: *"You enter quarantine presumed to have an unacceptable $5\%$ flake rate until you provide overwhelming empirical proof of stability."*

---

### 4.2 Quantitative Promotion & Demotion Gate Thresholds

Let $\theta_{\text{max}} = 0.01$ ($1.0\%$) denote the maximum acceptable production failure rate SLA. Rather than checking streak counters, the engine inspects the exact cumulative posterior distribution:

$$
P(\theta_t \le 0.01 \mid \alpha_t, \beta_t) = \int_0^{0.01} \frac{\theta^{\alpha_t - 1} (1 - \theta)^{\beta_t - 1}}{B(\alpha_t, \beta_t)} \, d\theta = I_{0.01}(\alpha_t, \beta_t)
$$

where $I_x(\alpha, \beta)$ is the regularized incomplete beta function.

#### 1. Promotion Criterion (Quarantine $\to$ Blocking)
A test is promoted from Quarantine to the Blocking Quality Gate if and only if the **$95\%$ credible upper bound** on its failure rate falls strictly below $\theta_{\text{max}} = 0.01$:

$$
P(\theta_t \le 0.01 \mid \alpha_t, \beta_t) \ge 0.95
$$

Equivalently, the 95th posterior quantile $Q_{0.95}(\text{Beta}(\alpha_t, \beta_t)) \le 0.01$. The pipeline demands $95\%$ certainty that the test causes fewer than 1 false alarm per 100 merges.

#### 2. Demotion Criterion (Blocking $\to$ Quarantine)
A test currently in the Blocking tier is immediately evicted and demoted back to Quarantine whenever the posterior probability that it exceeds the acceptable flake floor $\theta_{\text{max}} = 0.01$ reaches or exceeds $90\%$:

$$
P(\theta_t > 0.01 \mid \alpha_t, \beta_t) \ge 0.90 \iff P(\theta_t \le 0.01 \mid \alpha_t, \beta_t) \le 0.10
$$

#### Why Asymmetric Credible Thresholds Eliminate Both Fatal Traps:
* **Elimination of the False Promotion Trap**: Under streak rules, a $5\%$ flaky test easily flukes 30 consecutive passes ($21.5\%$ probability) and enters Blocking. Under the Bayesian criterion, even after 30 clean passes, the posterior confidence that $\theta \le 0.01$ is only $38.9\% \ll 95\%$. The test remains safely trapped in Quarantine.
* **Elimination of the Hair-Trigger Demotion Trap**: In streak systems, an exemplary, rock-solid test ($\theta = 0.001$) with 500 consecutive passes that suffers a single transient OS networking hiccup has its streak reset to 0 and is immediately flagged or demoted. Under the Bayesian filter, after $\beta = 299$ passes, a single failure updates the state to $\text{Beta}(2, 299)$. The posterior probability $P(\theta > 0.01 \mid 2, 299) \approx 19.8\% \ll 90\%$. The filter absorbs the transient blip without disrupting developers, while still slightly lowering confidence until subsequent passes confirm health.

---

### 4.3 Complete Worked Numerical Comparison: 5% Flaky Test Over 50 to 300 Runs

Let us trace a concrete side-by-side execution trace of a genuinely flaky test with true failure rate $\theta_{\text{true}} = 0.05$ evaluated under both gating architectures.

#### Gate 1: The Heuristic $(N=30, M=30)$ Streak Gate
* **The Math**: The probability that a test with $\theta = 0.05$ passes a single run is $1 - 0.05 = 0.95$.
* The probability of passing 30 runs consecutively is:

$$
P(\text{30 consecutive passes}) = (0.95)^{30} \approx \mathbf{21.46\%}
$$

* **The Operational Consequence**: In a high-velocity CI environment executing 10 to 15 test runs per day, the test enters a streak evaluation window every few days. Within 2 to 3 retry windows, the test inevitably rolls a lucky 30-pass streak.
* **The Failure**: The heuristic gate proclaims: *"30 passes achieved! Promoted to Blocking!"* Once in the blocking tier, this test breaks the main build once every 20 merges ($5\%$ failure rate), grinding the engineering organization to a halt.

#### Gate 2: The Bayesian Dynamic Gate ($\text{Beta}(1, 19)$ Prior, Target $P(\theta \le 0.01) \ge 0.95$)
* **Stage A: Lucky Streak of 30 Passes ($k=0, n=30$)**:
  * Prior: $\text{Beta}(\alpha_0 = 1, \beta_0 = 19)$.
  * After 30 consecutive passes with zero failures, the updated parameters are:

$$
\alpha_{30} = 1 + 0 = 1, \qquad \beta_{30} = 19 + 30 = 49 \implies \text{Beta}(1, 49)
$$

  * The posterior mean failure rate is:

$$
\mathbb{E}[\theta] = \frac{1}{1 + 49} = \frac{1}{50} = \mathbf{2.0\%}
$$

  * The posterior probability that the failure rate meets the target threshold $\theta \le 0.01$ is:

$$
P(\theta \le 0.01 \mid \alpha=1, \beta=49) = 1 - (1 - 0.01)^{49} = 1 - (0.99)^{49} \approx \mathbf{38.9\%}
$$

  * **The Verdict**: $38.9\% \ll 95\%$ required confidence! Despite 30 clean passes in a row, the Bayesian gate **refuses to promote**.
  * **Why?** Thirty passes is mathematically incapable of certifying a $<1\%$ failure rate at $95\%$ confidence. In closed form, certifying $\theta \le 0.01$ with zero observed failures requires:

$$
1 - (0.99)^\beta \ge 0.95 \implies (0.99)^\beta \le 0.05 \implies \beta \ge \frac{\ln(0.05)}{\ln(0.99)} \approx \mathbf{298\text{ effective passes}}
$$

* **Stage B: Long-Term Horizon (300 Runs)**:
  * Over 300 runs in Quarantine, a true $5\%$ flaky test generates an average of $300 \times 0.05 = 15$ failures and $285$ passes.
  * Incorporating the skeptical prior, the un-decayed posterior parameters reach:

$$
\alpha_{300} = 1 + 15 = 16, \qquad \beta_{300} = 19 + 285 = 304 \implies \text{Beta}(16, 304)
$$

  * The posterior distribution converges tightly around the true rate:

$$
\mathbb{E}[\theta] = \frac{16}{16 + 304} = \frac{16}{320} = \mathbf{5.0\%}
$$

  * The posterior confidence of satisfying the SLA collapses completely:

$$
P(\theta \le 0.01 \mid \alpha=16, \beta=304) \approx 2.17 \times 10^{-7} < \mathbf{10^{-5}}
$$

  * **The Verdict**: The Bayesian gate keeps the test **permanently quarantined**. The flaky test cannot graduate into Blocking until developers identify and fix the underlying concurrency race, deadlock, or leaked fixture.

---

> 🐍 **See the Code**: Simulate the 400-run promotion state machine in Python!  
> Open **[Python Chapter 20: Approach 2 & Simulation](../python/20_test_promotion_heuristics_vs_bayes.ipynb#3-how-the-course-solves-test-promotion)**.

---

## 5. Summary: The Production CI Promotion Playbook

### Architectural Comparison Matrix

| Evaluation Dimension | Naive Streak Heuristic ($(N=30, M=30)$) | Bayesian Dynamic Discount Gate ($\text{Beta}(1, 19), \gamma=0.98$) |
| :--- | :--- | :--- |
| **30 Consecutive Passes ($k=0, n=30$)** | **Falsely Promotes to Blocking** (meets $M=30$ condition). | **Rejects Promotion** ($P(\theta \le 0.01) = 38.9\% \ll 95\%$; requires $\approx 298$ passes). |
| **Single Transient Flake (after 300 passes)** | **Hair-Trigger Demotion** (streak resets instantly to 0; test evicted). | **Tolerant & Robust** ($P(\theta > 0.01) \approx 19.8\% \ll 90\%$; absorbs noise without thrash). |
| **Chronic 5% Flake (over 50–300 runs)** | **Vulnerable to Flukes** ($21.5\%$ promotion chance per window; $>60\%$ across 500 runs). | **Permanently Quarantined** ($P(\theta \le 0.01) < 10^{-5}$; posterior locks onto $5.0\%$). |
| **Sudden Regression (0.1% $\to$ 20% flake)** | **Slow to Detect** (only trips after enough runs to break moving windows). | **Rapid Demotion** (exponential decay forces $P(\theta > 0.01) \ge 90\%$ in $< 10$ runs). |
| **Evidence Weighting** | Binary amnesia (everything before streak is forgotten). | Continuous exponential weighting ($\gamma^t$ smooth memory). |
| **Computational Overhead** | $\mathcal{O}(1)$ integer counter. | $\mathcal{O}(1)$ closed-form conjugate update & Beta CDF. |

By replacing streak counters with dynamic Bayesian filters, engineering organizations eliminate build pipeline blockage, protect developer flow, and maintain rock-solid production quality gates.

---


---
## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 19: Verifying Flaky Fixes (SPRT & BF)](19_verifying_flaky_test_fixes.ipynb)** *(Part V: Production Reliability, CI/CD & Active Learning)*
* ➡️ **Next**: **[Ch 21: Framework Reliability Contracts](21_framework_reliability_and_tolerance_contracts.ipynb)** *(Part V: Production Reliability, CI/CD & Active Learning)*
* 🔄 **Companion Tracks for Chapter 20**: [📖 Conceptual](../conceptual/20_test_promotion_heuristics_vs_bayes.ipynb) | [🤖 AI-Native](../ai/20_test_promotion_heuristics_vs_bayes.ipynb) | [🐍 Python](../python/20_test_promotion_heuristics_vs_bayes.ipynb) | [🔵 R](../r/20_test_promotion_heuristics_vs_bayes.ipynb)
